<a href="https://colab.research.google.com/github/gibsonx/jlpt_simulator/blob/dev/graphs/n3/outliner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/gibsonx/jlpt_simulator.git
    %cd jlpt_simulator
    !git checkout dev
    !apt-get install python3-dev graphviz libgraphviz-dev pkg-config
    !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import json
import logging
import random
import time
import pandas as pd
import yaml
import inspect
from tqdm import tqdm
import os
from libs.Logger import logger
from datetime import datetime
from docx import Document
from html4docx import HtmlToDocx
import uuid
from libs.CosmosMongoDB import CosmosMongoDB
from libs.LLMs import *
from IPython.display import display, Markdown, HTML
import datetime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from libs.Utils import render_to_html,collect_vocabulary,_load_vocab_and_resources
from graphs.common.Schema import Outline
from langchain_core.prompts import ChatPromptTemplate
from graphs.common.ExamGenerator import ExamGenerator
load_dotenv()

from graphs.common.TaskRunner import TaskRunner

# N1 Level Exam

In [3]:
# runner = TaskRunner(level="N1", exam_type="fast_exam")
# n1_outline, n1_exam_paper = runner.run()

## N1 Outline Preview

In [4]:
# display(Markdown(n1_outline.as_str))

## N1 HTML Result

In [5]:
# html_output = render_to_html(n1_exam_paper['sections'])
# display(HTML(html_output))
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# filename = f"./output/jlpt_simulator/JLPT_{timestamp}.html"

# with open(filename, "w", encoding="utf-8") as file:
#     file.write(html_output)

# N2 Level Exam

In [6]:
# runner = TaskRunner(level="N2", exam_type="fast_exam")
# n2_outline, n2_exam_paper = runner.run()

## N2 Outline Preview

In [7]:
# display(Markdown(n2_outline.as_str))

## N2 Exam Result

In [8]:
# html_output = render_to_html(n2_exam_paper['sections'])
# display(HTML(html_output))

In [9]:
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# filename = f"./output/JLPT_N2_{timestamp}.html"
# 
# with open(filename, "w", encoding="utf-8") as file:
#     file.write(html_output)

# N3 Level Exam

In [10]:
runner = TaskRunner(level="N3", exam_type="fast_exam")
n3_outline, n3_exam_paper = runner.run()

2025-11-14 22:45:41,868 - INFO - jlpt - Module 'graphs.n3.outliner' imported successfully.
2025-11-14 22:45:41,868 - INFO - Module 'graphs.n3.outliner' imported successfully.
2025-11-14 22:45:41,870 - INFO - jlpt - Current Task ID None is being initialized
2025-11-14 22:45:41,870 - INFO - Current Task ID None is being initialized
2025-11-14 22:45:55,225 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-14 22:45:55,248 - INFO - jlpt - Outline of the exam:

# 日本語能力試験N3 模擬試験問題

## 第1部：語彙

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。1問。80%が難問。

- **嚴禁**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **ぎょうぎ**

### word_meaning

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **ふこう**

### synonym_substitution

問題4：意味が最も近いものを、1・2・3・4から一つえらびなさい。1問。

- **かわいそう**

### word_usage

問題5：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。1問。8

## N3 Outline Preview

In [11]:
display(Markdown(n3_outline.as_str))

# 日本語能力試験N3 模擬試験問題

## 第1部：語彙

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。1問。80%が難問。

- **嚴禁**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **ぎょうぎ**

### word_meaning

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **ふこう**

### synonym_substitution

問題4：意味が最も近いものを、1・2・3・4から一つえらびなさい。1問。

- **かわいそう**

### word_usage

問題5：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。1問。80%が難問。

- **誤解**

## 第2部：文法

### sentence_grammar

問題6：つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。1問。

- **健康診断や医者への訪問について話す**...てからでないと／...てからでなければ

### sentence_sort

問題7：つぎの文の ★ に入る最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **家事の分担について話す**...ようにする

### sentence_structure

問題8：つぎの文章を読んで、文章全体の内容を考えて、文中の 19 から 22 の中に入る最もよいものを、1・2・3・4から一つえらびなさい。1問。4つの文法項目を統合。

- **技術について話す**...によって／...ておく／...ようになる／...そうだ

## 第3部：読解

### short_passage_mail_read

問題1-1：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **友人へのプレゼント選びについて話す**

### short_passage_notification_read

問題1-2：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **公共施設の利用方法について話す**

### short_passage_narrative_read

問題1-3：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **家族について話す**

### midsize_passage_read

問題2：つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **週末の予定について話す**

### long_passage_read

問題3：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **環境問題について話す**

### info_retrieval

問題4：これを読んで、下の質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。情報検索。1記事。

- **領収書を求める**

## 第4部：聴解

### topic_understanding_txt

問題1-1：まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。1問。

- **レストランで食べ物を注文する**

### topic_understanding_img

問題1-2：まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。1問。

- **バスの時刻表を尋ねる**

### keypoint_understanding

問題2：まず質問を聞いてください。そのあと、問題用紙を見てください。読む時間があります。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。1問。

- **購入したい商品の説明**

### summary_understanding

問題3：問題用紙に何も印刷されていません。話の前に質問はありません。まず話を聞いてください。それから、質問と選択肢を聞いて、1から4の中から最もよいものを一つえらんでください。1問。

- **旅行の計画について話す**

### active_expression

問題4：元を見ながら質問を聞いてください。やじるし（➔）の人は何と言いますか。1から3の中から、最もよいものを一つえらんでください。1問。

- **道を尋ねる**

### immediate_ack

問題5：問題用紙に何も印刷していません。まず文を聞いてください。それから、その返事を聞いて、1から3の中から、最もよいものを一つえらんでください。1問。

- **おすすめを尋ねる**

## N3 Exam Result

In [12]:
html_output = render_to_html(n3_exam_paper['sections'])
display(HTML(html_output))

In [13]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N3_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N4 Level Exam

In [14]:
# runner = TaskRunner(level="N4", exam_type="fast_exam")
# n4_outline, n4_exam_paper = runner.run()

## N4 Outline Preview

In [15]:
# display(Markdown(n4_outline.as_str))

## N4 Exam Result

In [16]:
# html_output = render_to_html(n4_exam_paper['sections'])
# display(HTML(html_output))

# N5 Level Exam

In [17]:
# runner = TaskRunner(level="N5", exam_type="fast_exam")
# n5_outline, n5_exam_paper = runner.run()

## N5 Outline Preview

In [18]:
# display(Markdown(n5_outline.as_str))

## N5  Exam Result

In [19]:
# html_output = render_to_html(n5_exam_paper['sections'])
# display(HTML(html_output))